# 🔄 Cập nhật dữ liệu FAQ & Embeddings (update.ipynb)

Notebook này giúp bạn **cập nhật dữ liệu** và **đồng bộ hoá embeddings TF‑IDF** vào **ChromaDB** theo 2 chế độ:
- **APPEND**: Giữ nguyên `tfidf_model.pkl` hiện tại → **chỉ thêm** các bản ghi mới vào ChromaDB.
- **REBUILD**: **Huấn luyện lại** TF‑IDF trên **toàn bộ dữ liệu** → Xoá collection cũ và **build lại toàn bộ** embeddings trong ChromaDB.

> ⚠️ Lưu ý: Nếu bạn **thay đổi từ điển** (vocab) hoặc thêm nhiều từ mới, nên dùng **REBUILD** để đảm bảo truy vấn nhất quán.


In [80]:
# ===========================
# 1) CẤU HÌNH & THƯ VIỆN
# ===========================
import os, pickle
import pandas as pd
import numpy as np
import chromadb
from manual_tfidf import ManualTfidfVectorizer

# Thử import xldl để tiền xử lý
try:
    import xldl
    def preprocess_text(s: str) -> str:
        return xldl.main(s)
except Exception:
    def preprocess_text(s: str) -> str:
        return s

# ==== CÁC THAM SỐ CẦN SỬA CHO PHÙ HỢP DỰ ÁN CỦA BẠN ====
CSV_PATH = "cauhoi_update.csv"                 # File dữ liệu nguồn
TEXT_COL = "pre_question"            # Cột văn bản đã tiền xử lý
ID_COL = None                        # Nếu có cột ID riêng trong CSV, điền tên cột; nếu None, sẽ tự sinh doc_0, doc_1,...

CHROMA_PATH = "./chroma_db"          # Thư mục DB Chroma
COLLECTION_NAME = "faq_tfidf"        # Tên collection
TFIDF_PKL = "tfidf_model.pkl"        # File lưu/đọc TF-IDF model

# Chế độ cập nhật: 'APPEND' hoặc 'REBUILD'
MODE = "APPEND"  # hoặc "REBUILD"

# Nếu MODE='APPEND', giới hạn tối thiểu độ dài văn bản để coi là "mới"
MIN_TEXT_LEN = 1

print(f"CSV_PATH         = {CSV_PATH}")
print(f"TEXT_COL         = {TEXT_COL}")
print(f"ID_COL           = {ID_COL}")
print(f"CHROMA_PATH      = {CHROMA_PATH}")
print(f"COLLECTION_NAME  = {COLLECTION_NAME}")
print(f"TFIDF_PKL        = {TFIDF_PKL}")
print(f"MODE             = {MODE}")


CSV_PATH         = cauhoi_update.csv
TEXT_COL         = pre_question
ID_COL           = None
CHROMA_PATH      = ./chroma_db
COLLECTION_NAME  = faq_tfidf
TFIDF_PKL        = tfidf_model.pkl
MODE             = APPEND


In [81]:
assert os.path.exists(CSV_PATH), f"Không tìm thấy file CSV: {CSV_PATH}"
df = pd.read_csv(CSV_PATH)

In [82]:
df["pre_question"] = df["question"].astype(str).apply(xldl.main)

In [83]:
# ===========================
# 2) NẠP DỮ LIỆU CSV
# ===========================


# Nếu bạn muốn tự sinh TEXT_COL từ cột 'question' bằng xldl.main(), bật phần sau:
if TEXT_COL not in df.columns and "question" in df.columns:
    print(f"TEXT_COL '{TEXT_COL}' chưa có. Tạo từ cột 'question' bằng xldl.main()...")
    df[TEXT_COL] = df["question"].astype(str).fillna("").map(preprocess_text)

# Nếu TEXT_COL đã tồn tại nhưng bạn muốn tái xử lý, bỏ comment dòng dưới:
# df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("").map(preprocess_text)

# Lọc rỗng
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("")
df = df[df[TEXT_COL].str.len() >= MIN_TEXT_LEN].copy()

print("Kích thước dữ liệu:", df.shape)
df.head(3)


Kích thước dữ liệu: (32, 4)


,STT,question,answer,pre_question
0,1,Làm thế nào để đăng ký học phần cho học kỳ đầu...,Sinh viên đăng nhập vào hệ thống UD-Portal hoặ...,làm đăng_ký học học_kỳ đầu_tiên
1,2,Làm sao để biết lịch học và lịch thi của mình?,"Vào cổng đào tạo → “Tra cứu lịch học, lịch thi...",làm lịch học lịch thi mình
2,3,Nếu bị trùng lịch hoặc sai môn khi đăng ký học...,Liên hệ Phòng Đào tạo hoặc cố vấn học tập tron...,nếu trùng lịch sai môn đăng_ký học xử_lý nào


In [84]:
# ===========================
# 3) KẾT NỐI CHROMADB
# ===========================
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_or_create_collection(COLLECTION_NAME, metadata={"hnsw:space":"cosine"}, embedding_function=None)

print("Số bản ghi hiện có trong collection:", collection.count())


Số bản ghi hiện có trong collection: 31


In [85]:
# ===========================
# 4) CHUẨN BỊ ID, DOC, META
# ===========================
def build_payloads(df: pd.DataFrame, text_col: str, id_col=None):
    docs = df[text_col].astype(str).fillna("").tolist()
    if id_col is not None and id_col in df.columns:
        ids = df[id_col].astype(str).tolist()
    else:
        ids = [f"doc_{i}" for i in range(len(df))]
    # metadata: bỏ cột text và id
    drop_cols = [c for c in [text_col, id_col] if c is not None and c in df.columns]
    metas = df.drop(columns=drop_cols, errors="ignore").to_dict(orient="records")
    return ids, docs, metas

ids_all, docs_all, metas_all = build_payloads(df, TEXT_COL, ID_COL)
len(ids_all), len(docs_all), len(metas_all)


(32, 32, 32)

In [86]:
# ===========================
# 5A) CHẾ ĐỘ APPEND
# ===========================
if MODE.upper() == "APPEND":
    assert os.path.exists(TFIDF_PKL), "APPEND yêu cầu đã có tfidf_model.pkl. Hãy chạy REBUILD lần đầu."
    with open(TFIDF_PKL, "rb") as f:
        tfidf = pickle.load(f)

    # Lấy danh sách ID đã tồn tại để chỉ thêm cái mới
    try:
        existing = set(collection.get(ids=ids_all)["ids"])
    except Exception:
        existing = set()

    new_rows = [(i,d,m) for i,d,m in zip(ids_all, docs_all, metas_all) if i not in existing]

    if not new_rows:
        print("Không có bản ghi mới để thêm (theo ID).")
    else:
        new_ids, new_docs, new_metas = zip(*new_rows)
        X_new = tfidf.transform(list(new_docs))
        embeddings = [row.tolist() for row in X_new]
        collection.add(ids=list(new_ids), documents=list(new_docs), metadatas=list(new_metas), embeddings=embeddings)
        print(f"✅ Đã thêm mới {len(new_ids)} bản ghi vào ChromaDB.")
    print("Tổng số bản ghi hiện tại:", collection.count())


✅ Đã thêm mới 1 bản ghi vào ChromaDB.
Tổng số bản ghi hiện tại: 32


In [87]:
# ===========================
# 5B) CHẾ ĐỘ REBUILD
# ===========================
if MODE.upper() == "REBUILD":
    texts = [str(t) for t in docs_all]
    tfidf = ManualTfidfVectorizer()
    X_all = tfidf.fit_transform(texts)

    # Lưu lại model TF-IDF
    with open(TFIDF_PKL, "wb") as f:
        pickle.dump(tfidf, f)
    print("✅ Đã lưu lại TF-IDF model:", TFIDF_PKL)

    # Xoá collection cũ và tạo lại sạch
    try:
        client.delete_collection(COLLECTION_NAME)
        print("🗑️ Đã xoá collection cũ.")
    except Exception:
        pass
    collection = client.get_or_create_collection(COLLECTION_NAME, metadata={"hnsw:space":"cosine"}, embedding_function=None)

    embeddings = [row.tolist() for row in X_all]
    collection.add(ids=ids_all, documents=docs_all, metadatas=metas_all, embeddings=embeddings)
    print(f"✅ Đã build lại toàn bộ {len(ids_all)} bản ghi vào ChromaDB.")
    print("Tổng số bản ghi hiện tại:", collection.count())


In [88]:
# ===========================
# 6) KIỂM TRA NHANH TRUY VẤN
# ===========================
def embed_query(q: str):
    # dùng model hiện có (đã load ở APPEND hoặc vừa train ở REBUILD)
    return tfidf.transform([q]).astype(float)[0].tolist()

sample_query = "Trường có giảng viên nào tận tâm và yêu thương sinh viên không?"
sample_query = xldl.main(sample_query)
qvec = embed_query(sample_query)
res = collection.query(query_embeddings=[qvec], n_results=3, include=["documents","metadatas","distances"])

for i,(doc,dist,meta) in enumerate(zip(res["documents"][0], res["distances"][0], res["metadatas"][0]),1):
    print(f"{i}. sim={(1-dist)*100:.1f}%  |  doc={doc[:80]}  |  meta={meta}")


1. sim=100.0%  |  doc=trường giảng_viên tận_tâm yêu_thương sinh_viên không  |  meta={'STT': 32, 'answer': 'Giảng viên Nguyễn Thành Thủy được biết đến là người rất tận tâm và yêu thương sinh viên.', 'question': 'Trường có giảng viên nào tận tâm và yêu thương sinh viên không?'}
2. sim=67.3%  |  doc=trường chương_trình hỗ_trợ sinh_viên khăn không  |  meta={'question': 'Trường có chương trình hỗ trợ sinh viên khó khăn không?', 'STT': 25, 'answer': 'Có. Sinh viên có thể nộp đơn xin hỗ trợ học phí hoặc quỹ “Tiếp sức đến trường” tại Phòng CTSV.'}
3. sim=53.2%  |  doc=trường môn_học không  |  meta={'question': 'Trường có môn học tự chọn không?', 'answer': 'Có. Sinh viên cần hoàn thành đủ số tín chỉ tự chọn theo chương trình đào tạo của từng ngành (thường 9–12 tín chỉ).', 'STT': 29}
